In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import datetime, timedelta
import re
import time
import uuid
import pandas as pd

def get_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=en-AU")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36")
    return webdriver.Chrome(options=options)


def safe(card, sel):
    els = card.find_elements(By.CSS_SELECTOR, sel)
    return els[0].text.strip() if els else "N/A"


def fix_encoding(text):
    """Fix mojibake from UTF-8 being misread as latin-1."""
    if not text or text == "N/A":
        return text
    try:
        return text.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def clean_url(url):
    """Strip tracking params — keep only the base job URL."""
    if not url:
        return None
    return url.split("?")[0]


def parse_posted_date(text):
    if not text or text == "N/A":
        return None
    today = datetime.today()
    text = text.lower().replace("posted", "").replace("ago", "").strip()
    match = re.match(r"(\d+)\s*(h|d|mo|w|m|y)", text)
    if not match:
        return None
    value, unit = int(match.group(1)), match.group(2)
    if unit == "h":
        date = today - timedelta(hours=value)
    elif unit == "d":
        date = today - timedelta(days=value)
    elif unit == "w":
        date = today - timedelta(weeks=value)
    elif unit in ("mo", "m"):
        date = today - timedelta(days=value * 30)
    elif unit == "y":
        date = today - timedelta(days=value * 365)
    else:
        return None
    return date.strftime("%Y-%m-%d")


def scrape_jora(query="data engineer", location="New South Wales"):
    driver = get_driver()
    url = f"https://au.jora.com/j?q={query.replace(' ', '+')}&l={location.replace(' ', '+')}&sfr=5"
    print(f"Fetching: {url}\n")
    driver.get(url)
    time.sleep(3)

    cards = driver.find_elements(By.CSS_SELECTOR, "div.job-card[data-job-card]")
    print(f"Found {len(cards)} jobs — collecting card data first...\n")

    # Step 1: collect all card data before navigating away
    card_data = []
    for card in cards:
        title_el = card.find_elements(By.CSS_SELECTOR, "h2.job-title a")
        card_data.append({
            "title":      title_el[0].text.strip() if title_el else "N/A",
            "company":    safe(card, "span.job-company"),
            "location":   safe(card, "a.job-location"),
            "posted_raw": safe(card, "span.job-listed-date"),
            "url":        clean_url(title_el[0].get_attribute("href")) if title_el else None,
        })

    # Step 2: visit each URL for description
    rows = []
    for i, job in enumerate(card_data):
        description = "N/A"
        if job["url"]:
            try:
                driver.get(job["url"])
                time.sleep(2)
                desc_el = driver.find_elements(By.CSS_SELECTOR, "div#job-description-container")
                raw_desc = desc_el[0].text.strip() if desc_el else "N/A"
                description = fix_encoding(raw_desc)
            except Exception as e:
                description = f"Error: {e}"

        rows.append({
            "job_id":      str(uuid.uuid4()),
            "title":       fix_encoding(job["title"]),
            "company":     fix_encoding(job["company"]),
            "location":    fix_encoding(job["location"]),
            "date_posted": parse_posted_date(job["posted_raw"]),
            "description": description,
            "job_url":     job["url"],
        })
        print(f"  [{i+1}/{len(card_data)}] ✓ {job['title']} @ {job['company']}")

    driver.quit()

    df = pd.DataFrame(rows, columns=["job_id", "title", "company", "location", "date_posted", "description", "job_url"])
    return df


# --- Run ---
df = scrape_jora()
print(f"\nDone! Shape: {df.shape}")
df.head()

Fetching: https://au.jora.com/j?q=data+engineer&l=New+South+Wales&sfr=5

Found 15 jobs — collecting card data first...

  [1/15] ✓ Data Engineer @ M&T Resources
  [2/15] ✓ Data Engineer @ XPT Software
  [3/15] ✓ Data Engineer / Permanent / $150,000 + super @ Allura Partners
  [4/15] ✓ Data Engineer (Graph Analytics) - EL 2 @ Australian Taxation Office
  [5/15] ✓ Data Engineer @ Endeavour Energy
  [6/15] ✓ Data Engineer @ Simplyai
  [7/15] ✓ Snowflake Data Engineer @ Mane Consulting
  [8/15] ✓ Data Engineer @ Brighte Capital
  [9/15] ✓ Data Engineer @ Skillerszone
  [10/15] ✓ Data Engineer @ Clickview
  [11/15] ✓ Data Engineer @ MYOB
  [12/15] ✓ Data Engineer - Snowflake (Admin + Delivery) @ Talenza
  [13/15] ✓ Data Engineer @ UBank
  [14/15] ✓ Data Engineer | Databricks @ Mane Consulting
  [15/15] ✓ Data Engineer - Sydney @ Ayan Infotech

Done! Shape: (15, 7)


,job_id,title,company,location,date_posted,description,job_url
0,dc002dbb-ecfc-4b4f-b5ea-c74250ab9984,Data Engineer,M&T Resources,Sydney CBD NSW,2026-03-20,I'm recruiting for a fast-growing technology c...,https://au.jora.com/job/Data-Engineer-701aafca...
1,714a9229-a385-4a8d-8da2-0f5550209f1f,Data Engineer,XPT Software,Sydney NSW,2026-03-17,· Must have strong working expe...,https://au.jora.com/job/Data-Engineer-9f317a0d...
2,bfb4eda5-fd5a-4491-b178-749255eecea6,"Data Engineer / Permanent / $150,000 + super",Allura Partners,Sydney NSW,2026-03-20,Data Engineer / Permanent / Sydney or Melbourn...,https://au.jora.com/job/Data-Engineer-c071d1c5...
3,ef49e401-a431-4add-911a-eb5613b3a44a,Data Engineer (Graph Analytics) - EL 2,Australian Taxation Office,New South Wales,2026-03-20,Skip to main content,https://au.jora.com/job/Data-Engineer-699ecda7...
4,4b3edc53-9486-489a-a424-c92a10aef6dc,Data Engineer,Endeavour Energy,Parramatta NSW,2026-03-04,There’s never been a more exciting time to sha...,https://au.jora.com/job/Data-Engineer-0a873776...


In [ ]:
df            # see full table
